In [1]:
import json


with open("../../Experiment/SystemEvaluation/Shelf/shelf_single_no_color.json", "r", encoding="utf-8") as f:
    tasks = json.load(f)

with open("../../Experiment/SystemEvaluation/Shelf/Shelf.json", "r", encoding="utf-8") as f:
    shelf_data = json.load(f)



print("===============Shelf Datas==============" )
print(shelf_data[:3])

print("===============Tasks==============" )
print(tasks[:3])



===============Shelf Datas==============
[{'id': '409f6dff-d2e2-43c8-8ba5-2492b3e2e5d6', 'name': 'Shelf Light 1', 'position': {'x': -2.01285267, 'y': 0.429263115, 'z': 3.560356}, 'distance_from_user': 4.112417, 'eye_centrality_score': 30.5167942}, {'id': '80e64f7b-0589-4de7-977a-9d189a66fd1d', 'name': 'Shelf Light 4', 'position': {'x': -0.02322777, 'y': 0.447263241, 'z': 3.40095115}, 'distance_from_user': 3.43031359, 'eye_centrality_score': 8.679442}, {'id': '7ad8857c-4547-4f42-bcc0-dc17bd8e6fee', 'name': 'Shelf Light 7', 'position': {'x': 1.37828112, 'y': 0.429263115, 'z': 3.28866482}, 'distance_from_user': 3.5915513, 'eye_centrality_score': 24.4718132}]
===============Tasks==============
[{'command': '一番左上のライトをつけて', 'target_type': 'single', 'has_color': False, 'target_devices': [{'device_name': 'Shelf Light 1', 'color': 'white'}]}, {'command': '右下のライトを点灯して', 'target_type': 'single', 'has_color': False, 'target_devices': [{'device_name': 'Shelf Light 9', 'color': 'white'}]}, {'command

In [13]:

from no_tool_agent_runner import getSpatialRunnerForEvaluation
from sr_app_types.no_tool_agent_types import State, FilterAgentType, FilterAgentOutput
from langchain_core.messages import ToolMessage, HumanMessage, SystemMessage



def build_state_from_task(task, shelf_data):
    command = task["command"]
    
    tool_selection = FilterAgentOutput(
        filter_type="no_filter",
        params={"description": "No filter applied, but candidate devices are provided."},
        reasoning="This is for evaluating spatial reasoning performance without filtering."
    )

    return State(
        user_prompt=command,
        filterAgent=FilterAgentType(
            devices=shelf_data,
            output_tool_selection=tool_selection
        )
    )
def evaluate_task(task, result, shelf_data):
    # id→nameマップを構築
    id_to_name = {d["id"]: d["name"] for d in shelf_data}
    pred_ids = {d["id"] for d in result["agent_output"]["devices"]}
    pred_names = {id_to_name[pid] for pid in pred_ids if pid in id_to_name}
    gt_names = {d["device_name"] for d in task["target_devices"]}
    return pred_names == gt_names
evaluation_results = {
    "total": len(tasks),
    "correct": 0,
    "accuracy": 0.0,
    "results": []  # 個別タスクの結果
}

runner = getSpatialRunnerForEvaluation()

for i, task in enumerate(tasks):
    state = build_state_from_task(task, shelf_data)
    result = runner.invoke(state)

    # 評価判定
    is_correct = evaluate_task(task, result, shelf_data)

    # id→nameマップで出力変換
    id_to_name = {d["id"]: d["name"] for d in shelf_data}
    pred_ids = {d["id"] for d in result["agent_output"]["devices"]}
    pred_names = {id_to_name[pid] for pid in pred_ids if pid in id_to_name}
    gt_names = {d["device_name"] for d in task["target_devices"]}

    # 結果格納
    evaluation_results["results"].append({
        "task_index": i,
        "command": task["command"],
        "predicted_names": sorted(pred_names),
        "ground_truth_names": sorted(gt_names),
        "is_correct": is_correct,
        "agent_output": result["agent_output"],
    })

    if is_correct:
        evaluation_results["correct"] += 1
        print(f"[✅] Task {i}: {task['command']}")
    else:
        print(f"[❌] Task {i}: {task['command']}")
        print(f"     🔸 Predicted: {sorted(pred_names)}")
        print(f"     🔹 Expected : {sorted(gt_names)}")

# Accuracy計算
evaluation_results["accuracy"] = evaluation_results["correct"] / evaluation_results["total"]

print(f"\n=== 📊 Accuracy: {evaluation_results['accuracy']:.2%} ({evaluation_results['correct']}/{evaluation_results['total']}) ===")


==========[SR AGENT NODE]=========
OUTPUT:  デバイスの位置を比較し、x軸で最も左かつy軸で最も高い位置にあるデバイスを選定しました。Shelf Light 1が最も左上に位置しているため、このデバイスを操作対象としました。
RESPONSE:  一番左上のライトをつけました。
[SR AGENT TIME ELAPSED] 3.6086 sec
[✅] Task 0: 一番左上のライトをつけて
==========[SR AGENT NODE]=========
OUTPUT:  『右下』という指示に基づき、x軸が正の方向、y軸が負の方向に位置するライトを選択しました。Shelf Light 9はxが最も大きく、yが最も小さいため、空間的に右下に位置していると判断されました。
RESPONSE:  右下のライトを点灯しました。
[SR AGENT TIME ELAPSED] 3.0885 sec
[✅] Task 1: 右下のライトを点灯して
==========[SR AGENT NODE]=========
OUTPUT:  デバイスはy座標に基づいて3つの段に分類されました。中央の段はy ≈ 0.44の範囲にあり、その中でeye_centrality_scoreが最も低いデバイスを中央として選択しました。
RESPONSE:  真ん中の段の中央のデバイスをオンにしました。
[SR AGENT TIME ELAPSED] 2.9635 sec
[❌] Task 2: 真ん中の段の中央のやつをオンにして
     🔸 Predicted: ['Shelf Light 4']
     🔹 Expected : ['Shelf Light 5']
==========[SR AGENT NODE]=========
OUTPUT:  与えられたデバイスの中から、y座標が最も高いものを上の段と判断しました。上の段に属するデバイスの中で、x座標が最も大きいものを右端のライトとして選択しました。
RESPONSE:  上の段の右端のライトをオンにしました。
[SR AGENT TIME ELAPSED] 3.3300 sec
[✅] Task 3: 上の段の右端のライトをお願い
==========[SR AGENT NODE]

In [18]:
for result in evaluation_results["results"]:
    if not result["is_correct"]:
        print(result["task_index"], result["command"],  "\n")

2 真ん中の段の中央のやつをオンにして 

7 中央上のライトをつけて 

8 棚の中央にあるライトを点灯して 

10 棚の右列真ん中のやつを点灯 

13 下段中央を点灯して 

14 真ん中の棚の右側をつけて 



In [19]:

from no_tool_agent_runner import getSpatialRunner
from sr_app_types.no_tool_agent_types import State, FilterAgentType, FilterAgentOutput
from langchain_core.messages import ToolMessage, HumanMessage, SystemMessage

runner = getSpatialRunner()
task_index = 1

command_prompt = tasks[task_index]["command"]
tool_selection = FilterAgentOutput(
    filter_type="none",
    params={"description": "No filter applied, but candidate devices are provided."},
    reasoning="No filter used — direct device reasoning"
)


input_state: State = State(
        user_prompt=command_prompt,
        filterAgent=FilterAgentType(devices=shelf_data, output_tool_selection=tool_selection)

    )
res = runner.invoke(input_state)
# ID → 名前マップを作成（shelf_dataに基づく）
id_to_name = {device["id"]: device["name"] for device in shelf_data}

# LLMが選択したデバイスのID
predicted_ids = {d["id"] for d in res["agent_output"]["devices"]}

# それに対応する名前
predicted_names = {id_to_name[pid] for pid in predicted_ids if pid in id_to_name}

# 正解（target）デバイス名
target_names = {d["device_name"] for d in tasks[task_index]["target_devices"]}

# 比較して出力
is_correct = predicted_names == target_names

print("=== コマンド ===")
print(command_prompt)
print("=== LLMが選んだデバイス ===")
print(predicted_names)
print("=== 正解デバイス ===")
print(target_names)
print("=== 判定結果 ===")
print("✅ 正解" if is_correct else "❌ 不正解")

==========[SR AGENT NODE]=========
OUTPUT:  デバイスの位置情報から、x軸が正の方向で、y軸が最も低い位置にあるデバイスを右下と判断しました。これに基づき、Shelf Light 9を選択しました。
RESPONSE:  右下のライトを点灯しました。
[SR AGENT TIME ELAPSED] 2.4660 sec

=====================[OPERATOR TOOL] operateDevice=====================
Devices to operate:  [DeviceControlData(id='0e98a772-fb03-4120-b64f-579eb5d5f696', state=True, intensity=100, color=RGBColor(r=255, g=255, b=255))]
Sending Operate Request to Test Server.
ERROR OCCURRED DURING OPERATION TOOL:  [WinError 10061] 対象のコンピューターによって拒否されたため、接続できませんでした。
=== コマンド ===
右下のライトを点灯して
=== LLMが選んだデバイス ===
{'Shelf Light 9'}
=== 正解デバイス ===
{'Shelf Light 9'}
=== 判定結果 ===
✅ 正解


In [18]:
# ID → 名前マップを作成（shelf_dataに基づく）
id_to_name = {device["id"]: device["name"] for device in shelf_data}

# LLMが選択したデバイスのID
predicted_ids = {d["id"] for d in res["agent_output"]["devices"]}

# それに対応する名前
predicted_names = {id_to_name[pid] for pid in predicted_ids if pid in id_to_name}

# 正解（target）デバイス名
target_names = {d["device_name"] for d in tasks[0]["target_devices"]}

# 比較して出力
is_correct = predicted_names == target_names

print("=== コマンド ===")
print(command_prompt)
print("=== LLMが選んだデバイス ===")
print(predicted_names)
print("=== 正解デバイス ===")
print(target_names)
print("=== 判定結果 ===")
print("✅ 正解" if is_correct else "❌ 不正解")


=== コマンド ===
一番左上のライトをつけて
=== LLMが選んだデバイス ===
{'Shelf Light 1'}
=== 正解デバイス ===
{'Shelf Light 1'}
=== 判定結果 ===
✅ 正解
